# HZMPS Simulation Study

**Model:** Bayesian state-space HZMPS with a sparse finite mixture over surveillance units.

**Design:** 6 scenarios × 30 replications = 180 fits.

| Factor | Levels |
|---|---|
| N | 50, 100, 200 |
| G_true | 3, 4 |
| Zero type | Inflation, Deflation |
| ρ_ζ | 0.3, 0.7 |

**Sampler:** 2 chains × (500 warmup + 500 sampling), 2 threads/chain, adapt_delta = 0.97, max_treedepth = 12, T = 50.

**Execution:** Save Version → Save & Run All. Runs up to 11 hours per commit. Resumes across commits from saved CSV.

In [ ]:
# SECTION 1: ENVIRONMENT SET UP
import os, sys, shutil, time, json, urllib.request
import numpy as np
import pandas as pd

# Install CmdStanPy and CmdStan
!pip install --upgrade cmdstanpy -q

CMDSTAN_DIR = '/kaggle/working/cmdstan-2.36.0'
if not os.path.exists(os.path.join(CMDSTAN_DIR, 'makefile')):
    print("Downloading CmdStan 2.36.0...")
    os.chdir('/kaggle/working')
    tgz_file = '/kaggle/working/cmdstan-2.36.0.tar.gz'
    tgz_url = ('https://github.com/stan-dev/cmdstan/releases/download/'
               'v2.36.0/colab-cmdstan-2.36.0.tgz')
    if not os.path.exists(tgz_file):
        urllib.request.urlretrieve(tgz_url, tgz_file)
    shutil.unpack_archive(tgz_file, '/kaggle/working')
    for item in os.listdir('/kaggle/working'):
        full = os.path.join('/kaggle/working', item)
        if os.path.isdir(full) and 'cmdstan-2.36.0' in item \
                and item != 'cmdstan-2.36.0':
            os.rename(full, CMDSTAN_DIR)
            break

os.environ['CMDSTAN'] = CMDSTAN_DIR

import cmdstanpy
print(f"CmdStan path:    {cmdstanpy.cmdstan_path()}")
print(f"CmdStan version: {cmdstanpy.cmdstan_version()}")

# --- Set up project directory ---
WORK_DIR = '/kaggle/working/hzmps_project'

def find_base_assets(root='/kaggle/input'):
    for root_dir, dirs, files in os.walk(root):
        if 'pivot_a_final.stan' in files:
            return os.path.dirname(root_dir)
    return None

if not os.path.exists(os.path.join(WORK_DIR, 'stan', 'pivot_a_final.stan')):
    base = find_base_assets()
    if base is None:
        raise FileNotFoundError(
            "Cannot find pivot_a_final.stan under /kaggle/input. "
            "Attach the hzmps-pivot-a dataset."
        )
    os.makedirs(WORK_DIR, exist_ok=True)
    for sub in ['stan', 'data']:
        s = os.path.join(base, sub)
        d = os.path.join(WORK_DIR, sub)
        if os.path.exists(s) and not os.path.exists(d):
            shutil.copytree(s, d)
    print(f"Copied project assets from {base}")
else:
    print(f"Project assets already in {WORK_DIR}")

os.chdir(WORK_DIR)
for sub in ['sim_results', 'figures']:
    os.makedirs(os.path.join(WORK_DIR, sub), exist_ok=True)

# --- Resume logic: search ALL of /kaggle/input for a progress CSV ---
def find_progress_file():
    """Look for any file matching backup_all_results* or all_results* under /kaggle/input."""
    candidates = []
    for root_dir, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if (f.startswith('backup_all_results') or
                f.startswith('all_results') or
                f == 'backup_all_results'):
                candidates.append(os.path.join(root_dir, f))
    if candidates:
        # Prefer the largest one (most rows)
        candidates.sort(key=lambda p: os.path.getsize(p), reverse=True)
        return candidates[0]
    return None

sim_results_dir = os.path.join(WORK_DIR, 'sim_results')
target_csv = os.path.join(sim_results_dir, 'all_results.csv')

progress = find_progress_file()
if progress:
    print(f"\nFound progress file: {progress}")
    try:
        df_prev = pd.read_csv(progress)
        df_prev.to_csv(target_csv, index=False)
        print(f"Resumed. Existing fits: {len(df_prev)}")
        print(f"Scenarios found: {sorted(df_prev['scenario_id'].unique().tolist())}")
    except Exception as e:
        print(f"Could not parse progress file: {e}")
        print("Starting fresh.")
else:
    print("\nNo progress file found. Starting fresh.")

print(f"\nWorking directory: {os.getcwd()}")
print(f"CPU cores: {os.cpu_count()}")

## Section 2: Stan Model with reduce_sum

Mathematical specification unchanged from the serial model. The likelihood loop over N×T observations is parallelized using `reduce_sum`.

In [ ]:
# SECTION 2: STAN MODEL WITH reduce_sum
import os, time
import cmdstanpy

stan_code = r"""
functions {
  real partial_sum_hzmps(array[] int y_slice,
                         int start, int end,
                         array[] int unit,
                         array[] int time,
                         vector omega,
                         matrix mu) {
    real lp = 0;
    for (k in 1:size(y_slice)) {
      int m = start + k - 1;
      int i = unit[m];
      int t = time[m];
      if (y_slice[k] == 0) {
        lp += log1m(omega[i]);
      } else {
        lp += log(omega[i])
              - mu[i, t]
              + y_slice[k] * log(mu[i, t])
              - lgamma(y_slice[k] + 1)
              - log1m_exp(-mu[i, t]);
      }
    }
    return lp;
  }
}

data {
  int<lower=1> N;
  int<lower=1> T;
  int<lower=1> G;
  array[N*T] int<lower=0> y;
  array[N*T] int<lower=1> unit;
  array[N*T] int<lower=1> time;
  real<lower=0> a_dirichlet;
  real<lower=0> delta_min;
  int<lower=1> grainsize;
}

transformed data {
  int<lower=0> M = N * T;
}

parameters {
  real mu_eta1;
  vector[G-1] gamma_raw;
  real<lower=0> sigma_eta;
  vector[N] eta;
  simplex[G] w;

  real beta0_mu;
  real<lower=-0.99, upper=0.99> rho_zeta;
  real<lower=0> sigma_zeta;
  matrix[N, T] zeta_raw;
}

transformed parameters {
  ordered[G] mu_eta;
  mu_eta[1] = mu_eta1;
  for (g in 2:G) {
    mu_eta[g] = mu_eta[g-1] + delta_min + log1p_exp(gamma_raw[g-1]);
  }

  matrix[N, T] zeta;
  for (i in 1:N) {
    zeta[i, 1] = zeta_raw[i, 1] * sigma_zeta / sqrt(1 - square(rho_zeta));
    for (t in 2:T)
      zeta[i, t] = rho_zeta * zeta[i, t-1] + zeta_raw[i, t] * sigma_zeta;
  }

  vector[N] omega;
  for (i in 1:N) omega[i] = inv_logit(eta[i]);

  matrix[N, T] mu;
  for (i in 1:N)
    for (t in 1:T)
      mu[i, t] = exp(beta0_mu + zeta[i, t]);
}

model {
  mu_eta1     ~ normal(0, 2);
  gamma_raw   ~ normal(-1, 1);
  sigma_eta   ~ normal(0, 0.3);
  w           ~ dirichlet(rep_vector(a_dirichlet / G, G));

  beta0_mu    ~ normal(0, 1);
  rho_zeta    ~ uniform(-0.99, 0.99);
  sigma_zeta  ~ normal(0, 0.3);

  to_vector(zeta_raw) ~ std_normal();

  for (i in 1:N) {
    vector[G] lp;
    for (g in 1:G)
      lp[g] = log(w[g]) + normal_lpdf(eta[i] | mu_eta[g], sigma_eta);
    target += log_sum_exp(lp);
  }

  target += reduce_sum(partial_sum_hzmps, y, grainsize,
                       unit, time, omega, mu);
}

generated quantities {
  vector[M] log_lik;
  matrix[N, G] cluster_prob;

  for (m in 1:M) {
    int i = unit[m];
    int t = time[m];
    if (y[m] == 0) {
      log_lik[m] = log1m(omega[i]);
    } else {
      log_lik[m] = log(omega[i])
                 - mu[i, t]
                 + y[m] * log(mu[i, t])
                 - lgamma(y[m] + 1)
                 - log1m_exp(-mu[i, t]);
    }
  }

  for (i in 1:N) {
    vector[G] lp;
    for (g in 1:G)
      lp[g] = log(w[g]) + normal_lpdf(eta[i] | mu_eta[g], sigma_eta);
    real lse = log_sum_exp(lp);
    for (g in 1:G)
      cluster_prob[i, g] = exp(lp[g] - lse);
  }
}
"""

stan_path = os.path.join(WORK_DIR, 'stan', 'pivot_a_rs.stan')
with open(stan_path, 'w') as f:
    f.write(stan_code)
print(f"Stan model written: {stan_path} ({os.path.getsize(stan_path)} bytes)")

print("\nCompiling with STAN_THREADS=true...")
t0 = time.time()
model_rs = cmdstanpy.CmdStanModel(
    stan_file=stan_path,
    cpp_options={'STAN_THREADS': 'true'},
    force_compile=False,
)
print(f"Compiled in {time.time() - t0:.1f} s")
print(f"Executable: {model_rs.exe_file}")

## Section 3: Helper Functions

`simulate_data()`, `fit_model()`, `extract_summary()` — used by the main loop below.

In [ ]:
# SECTION 3: HELPER FUNCTIONS
import os, json
import numpy as np
import pandas as pd


def simulate_data(N, T, G_true, w_true, mu_eta, sigma_eta,
                  beta0_omega, beta0_mu, rho_zeta, sigma_zeta,
                  seed, zero_type="infl"):
    rng = np.random.default_rng(seed)

    beta0_omega_eff = beta0_omega + 3.0 if zero_type == "defl" else beta0_omega

    delta_true = rng.choice(G_true, size=N, p=w_true)

    eta = np.zeros(N)
    for g in range(G_true):
        mask = (delta_true == g)
        eta[mask] = rng.normal(mu_eta[g], sigma_eta, size=mask.sum())

    zeta = np.zeros((N, T))
    zeta[:, 0] = rng.normal(0, sigma_zeta / np.sqrt(1 - rho_zeta**2), size=N)
    for t in range(1, T):
        zeta[:, t] = rho_zeta * zeta[:, t-1] + rng.normal(0, sigma_zeta, size=N)

    omega_per_unit = 1.0 / (1.0 + np.exp(-(beta0_omega_eff + eta)))
    omega = np.tile(omega_per_unit[:, None], (1, T))
    mu = np.exp(beta0_mu + zeta)

    Y = np.zeros((N, T), dtype=int)
    U = rng.uniform(size=(N, T))
    for i in range(N):
        for t in range(T):
            if U[i, t] > omega[i, t]:
                Y[i, t] = 0
            else:
                while True:
                    y = rng.poisson(mu[i, t])
                    if y > 0:
                        Y[i, t] = y
                        break

    stan_data = {
        'N': int(N), 'T': int(T), 'G': 5,
        'y': Y.flatten().tolist(),
        'unit': np.repeat(np.arange(1, N+1), T).tolist(),
        'time': np.tile(np.arange(1, T+1), N).tolist(),
        'a_dirichlet': 0.25,
        'delta_min': 0.3,
        'grainsize': max(1, int(N * T / 8)),
    }

    true_params = {
        'N': int(N), 'T': int(T), 'G_true': int(G_true),
        'zero_type': zero_type,
        'w_true': w_true.tolist(),
        'mu_eta': mu_eta.tolist(),
        'sigma_eta': float(sigma_eta),
        'beta0_omega_eff': float(beta0_omega_eff),
        'beta0_mu': float(beta0_mu),
        'rho_zeta': float(rho_zeta),
        'sigma_zeta': float(sigma_zeta),
        'zero_fraction': float((Y == 0).mean()),
        'cluster_sizes': np.bincount(delta_true, minlength=G_true).tolist(),
    }

    return {
        'Y': Y, 'delta_true': delta_true,
        'omega': omega, 'mu': mu,
        'true_params': true_params, 'stan_data': stan_data,
    }


def fit_model(model, stan_data, output_dir, seed=1,
              chains=2, iter_warmup=500, iter_sampling=500,
              threads_per_chain=2, adapt_delta=0.97,
              max_treedepth=12):
    os.makedirs(output_dir, exist_ok=True)
    return model.sample(
        data=stan_data,
        chains=chains, parallel_chains=chains,
        threads_per_chain=threads_per_chain,
        iter_warmup=iter_warmup, iter_sampling=iter_sampling,
        seed=seed, adapt_delta=adapt_delta, max_treedepth=max_treedepth,
        show_progress=False,
        output_dir=output_dir,
    )


def extract_summary(fit, true_params, delta_true, output_dir,
                    write_csv=True):
    summary = fit.summary()

    try:
        mv = fit.method_variables()
        n_div = int(np.sum(mv['divergent__']))
        n_td  = int(np.sum(mv['treedepth__']))
        n_tot = mv['divergent__'].size
        div_rate = n_div / n_tot
        td_rate  = n_td / n_tot
    except Exception:
        div_rate = float('nan')
        td_rate  = float('nan')

    scalar_names = ['sigma_eta', 'beta0_mu', 'rho_zeta', 'sigma_zeta']
    scalars = {}
    for p in scalar_names:
        if p in summary.index:
            row = summary.loc[p]
            scalars[p] = {
                'mean': float(row['Mean']),
                'rhat': float(row['R_hat']),
                'ess':  float(row['ESS_bulk']),
            }

    cluster_prob = fit.stan_variable('cluster_prob').mean(axis=0)
    map_assign = cluster_prob.argmax(axis=1) + 1
    G_max = cluster_prob.shape[1]
    G_true = len(true_params['mu_eta'])
    conf = np.zeros((G_true, G_max), dtype=int)
    for i in range(len(delta_true)):
        conf[delta_true[i], map_assign[i] - 1] += 1

    matched = {g: int(np.argmax(conf[g])) + 1 for g in range(G_true)}
    correct = sum(conf[g, matched[g]-1] for g in range(G_true))
    accuracy = correct / len(delta_true)

    w_draws = fit.stan_variable('w')
    G0 = (w_draws > 0.05).sum(axis=1)
    G0_mode = int(np.bincount(G0, minlength=G_max+1)[1:].argmax() + 1)
    G0_correct = int(G0_mode == true_params['G_true'])

    w_means = w_draws.mean(axis=0).tolist()

    result = {
        'divergence_rate': div_rate,
        'treedepth_rate':  td_rate,
        'scalars': scalars,
        'accuracy': float(accuracy),
        'G0_mode': G0_mode,
        'G0_correct': G0_correct,
        'w_means': w_means,
        'cluster_sizes_true': true_params['cluster_sizes'],
        'confusion': conf.tolist(),
    }

    if write_csv:
        flat = {
            'divergence_rate': div_rate,
            'treedepth_rate': td_rate,
            'accuracy': accuracy,
            'G0_mode': G0_mode,
            'G0_correct': G0_correct,
        }
        for p, vals in scalars.items():
            flat[f'{p}_mean'] = vals['mean']
            flat[f'{p}_rhat'] = vals['rhat']
            flat[f'{p}_ess']  = vals['ess']
        for g, wm in enumerate(w_means):
            flat[f'w{g+1}'] = wm
        pd.DataFrame([flat]).to_csv(
            os.path.join(output_dir, 'summary.csv'), index=False
        )

    return result


print("Functions loaded: simulate_data, fit_model, extract_summary")

## Section 4: Main Simulation Loop

6 scenarios × 30 reps = 180 fits. Budget: 11 hours per commit. Resumes across commits via `all_results.csv`.

In [ ]:
# SECTION 4: MAIN SIMULATION LOOP
import os, time, json
import numpy as np
import pandas as pd

SCENARIOS = [
    dict(id=1, N=50,  G_true=3, zero_type="infl", rho_zeta=0.7,
         description="Small N, standard inflation"),
    dict(id=2, N=100, G_true=3, zero_type="infl", rho_zeta=0.7,
         description="Medium N reference"),
    dict(id=3, N=200, G_true=3, zero_type="infl", rho_zeta=0.7,
         description="Large N consistency"),
    dict(id=4, N=100, G_true=4, zero_type="infl", rho_zeta=0.7,
         description="Larger true cluster count"),
    dict(id=5, N=100, G_true=3, zero_type="defl", rho_zeta=0.7,
         description="Zero-deflation"),
    dict(id=6, N=100, G_true=3, zero_type="infl", rho_zeta=0.3,
         description="Weak temporal dependence"),
]

N_REPS      = 30
T_FIXED     = 50
SIGMA_ETA   = 0.30
BETA0_OMEGA = -0.5
BETA0_MU    = 0.5
SIGMA_ZETA  = 0.40

MU_ETA = {
    2: np.array([-1.0, 1.0]),
    3: np.array([-1.5, 0.0, 1.5]),
    4: np.array([-2.0, -0.5, 1.0, 2.5]),
}
W_TRUE = {
    2: np.array([0.50, 0.50]),
    3: np.array([0.40, 0.35, 0.25]),
    4: np.array([0.30, 0.30, 0.25, 0.15]),
}

SIM_DIR    = os.path.join(WORK_DIR, 'sim_results')
MASTER_CSV = os.path.join(SIM_DIR, 'all_results.csv')
ROOT_CSV   = '/kaggle/working/backup_all_results.csv'
FAIL_LOG   = os.path.join(SIM_DIR, 'failures.log')
os.makedirs(SIM_DIR, exist_ok=True)

# Resume
if os.path.exists(MASTER_CSV):
    df_done = pd.read_csv(MASTER_CSV)
    done_keys = set(zip(df_done['scenario_id'], df_done['rep']))
    print(f"Resuming. {len(df_done)} fits already completed.")
else:
    df_done = pd.DataFrame()
    done_keys = set()
    print("Starting fresh.")

SESSION_BUDGET_HOURS = 11.0
session_start = time.time()
total_planned = len(SCENARIOS) * N_REPS
total_done    = len(done_keys)

if total_done >= total_planned:
    print(f"All {total_planned} fits complete. Nothing to do.")
else:
    print(f"Remaining fits: {total_planned - total_done}\n")
    stop_all = False

    for scen in SCENARIOS:
        if stop_all:
            break
        sid, N, G = scen['id'], scen['N'], scen['G_true']
        ztype, rho = scen['zero_type'], scen['rho_zeta']

        for rep in range(1, N_REPS + 1):
            if (sid, rep) in done_keys:
                continue

            print(f"\n{'='*72}")
            print(f"Scenario {sid} ({scen['description']}) | "
                  f"N={N}, G={G}, type={ztype}, rho={rho} | rep {rep}/{N_REPS}")
            print(f"Progress: {total_done}/{total_planned} "
                  f"({100*total_done/total_planned:.1f}%)")
            print(f"{'='*72}")

            seed_sim = 100_000 * sid + rep

            try:
                sim = simulate_data(
                    N=N, T=T_FIXED, G_true=G,
                    w_true=W_TRUE[G], mu_eta=MU_ETA[G],
                    sigma_eta=SIGMA_ETA,
                    beta0_omega=BETA0_OMEGA, beta0_mu=BETA0_MU,
                    rho_zeta=rho, sigma_zeta=SIGMA_ZETA,
                    seed=seed_sim, zero_type=ztype,
                )
            except Exception as e:
                with open(FAIL_LOG, 'a') as f:
                    f.write(f"Sim fail s{sid} r{rep}: {e}\n")
                print(f"Sim failed: {e}")
                continue

            rep_out = os.path.join(SIM_DIR, f"scen{sid}_rep{rep:03d}")
            os.makedirs(rep_out, exist_ok=True)

            t0 = time.time()
            try:
                fit = fit_model(
                    model=model_rs, stan_data=sim['stan_data'],
                    output_dir=rep_out, seed=seed_sim,
                    chains=2, iter_warmup=500, iter_sampling=500,
                    threads_per_chain=2,
                    adapt_delta=0.97, max_treedepth=12,
                )
            except Exception as e:
                with open(FAIL_LOG, 'a') as f:
                    f.write(f"Fit fail s{sid} r{rep}: {e}\n")
                print(f"Fit failed: {e}")
                continue
            t_fit = time.time() - t0

            try:
                res = extract_summary(
                    fit=fit, true_params=sim['true_params'],
                    delta_true=sim['delta_true'],
                    output_dir=rep_out, write_csv=True,
                )
            except Exception as e:
                with open(FAIL_LOG, 'a') as f:
                    f.write(f"Extract fail s{sid} r{rep}: {e}\n")
                print(f"Extract failed: {e}")
                continue

            row = {
                'scenario_id': sid, 'rep': rep,
                'N': N, 'T': T_FIXED, 'G_true': G,
                'zero_type': ztype, 'rho_zeta_true': rho,
                'sigma_eta_true': SIGMA_ETA,
                'beta0_mu_true': BETA0_MU,
                'sigma_zeta_true': SIGMA_ZETA,
                'divergence_rate': res['divergence_rate'],
                'treedepth_rate': res['treedepth_rate'],
                'accuracy': res['accuracy'],
                'G0_mode': res['G0_mode'],
                'G0_correct': res['G0_correct'],
                'fit_time_min': t_fit / 60,
            }
            for p, vals in res['scalars'].items():
                row[f'{p}_mean'] = vals['mean']
                row[f'{p}_rhat'] = vals['rhat']
                row[f'{p}_ess']  = vals['ess']
            for g, wm in enumerate(res['w_means']):
                row[f'w{g+1}'] = wm

            df_done = pd.concat([df_done, pd.DataFrame([row])],
                                ignore_index=True)
            df_done.to_csv(MASTER_CSV, index=False)
            df_done.to_csv(ROOT_CSV,   index=False)

            done_keys.add((sid, rep))
            total_done += 1

            print(f"Completed in {t_fit/60:.2f} min. "
                  f"Accuracy={res['accuracy']:.2f}, "
                  f"G0={res['G0_mode']}, "
                  f"div={res['divergence_rate']:.3f}")

            elapsed_h = (time.time() - session_start) / 3600
            if elapsed_h > SESSION_BUDGET_HOURS:
                print(f"\nSession budget {SESSION_BUDGET_HOURS} h reached.")
                stop_all = True
                break

    print(f"\n{'='*72}")
    print(f"Session ended. Completed: {total_done}/{total_planned} "
          f"({100*total_done/total_planned:.1f}%)")
    print(f"Master CSV: {MASTER_CSV}")
    print(f"Root CSV:   {ROOT_CSV}")

In [ ]:
# BACKUP: Zip results to /kaggle/working/ root
import os, shutil

sim_dir  = os.path.join(WORK_DIR, 'sim_results')
zip_path = '/kaggle/working/sim_results_backup'

if os.path.exists(zip_path + '.zip'):
    os.remove(zip_path + '.zip')

if os.path.exists(sim_dir):
    shutil.make_archive(zip_path, 'zip', root_dir=sim_dir)
    size_mb = os.path.getsize(zip_path + '.zip') / 1e6
    print(f"Backup zip: {zip_path}.zip ({size_mb:.2f} MB)")

root_csv = '/kaggle/working/backup_all_results.csv'
if os.path.exists(root_csv):
    print(f"CSV at root: {root_csv}")

print("\nDownload from the Output tab in the right panel.")

## Committing This Notebook For Reproducibility

**To run:**
1. Top-right: click **Save Version**.
2. Choose **Save & Run All (Commit)**.
3. Add a version note (e.g., "Run 1").
4. Click **Save**.
5. **Close the browser tab.** Kaggle runs the notebook in the background.
6. Come back in up to 12 hours.

**After a run completes:**
1. Open this notebook.
2. Right panel → **Output** tab.
3. Download `sim_results_backup.zip` and `backup_all_results.csv`.
4. These are your checkpoints.

**To resume:**
1. Re-attach the previous run's `backup_all_results.csv` as a dataset input, OR rely on `/kaggle/working/` being restored from the previous version.
2. Click **Save Version → Save & Run All (Commit)** again.
3. Cell B searches for the previous CSV and Cell H resumes from where it stopped.

**Expected runs: 4–5 total.**

Verify completion: Cell H prints `Completed: 180/180 (100.0%)`.